In [2]:
import numpy as np

In [11]:
x = 3
c = 2
sig = 1
op = 0.6065

In [3]:
def gauss(x, c, sigma=1.0):
    squared_norm = np.linalg.norm(x - c)**2
    return np.exp(-squared_norm / (2 * sigma**2))

In [18]:
def multi_quad(x,c,sig):
    return np.sqrt(np.linalg.norm(x - c)**2 + sig**2)

In [19]:
def inv_multi_quad(x,c,sig):
    return 1.0 / multi_quad(x,c,sig)

In [20]:
gauss(x,c,sig),multi_quad(x,c,sig),inv_multi_quad(x,c,sig)

(np.float64(0.6065306597126334),
 np.float64(1.4142135623730951),
 np.float64(0.7071067811865475))

In [ ]:
x = np.array([2,1,3])
c= np.array([1,2,2])
sig= 1

In [24]:
gauss(x,c,sig),multi_quad(x,c,sig),inv_multi_quad(x,c,sig)

(np.float64(0.22313016014842987),
 np.float64(1.9999999999999998),
 np.float64(0.5000000000000001))

# Heart

In [25]:
import pandas as pd

In [26]:
raw_df = pd.read_csv("/kaggle/input/datasets/johnsmith88/heart-disease-dataset/heart.csv")
X_df = raw_df.drop(["target"], axis=1)
Y_df = raw_df['target']

In [28]:
X = X_df.to_numpy()
Y = Y_df.to_numpy()

In [30]:
mu = np.mean(X, axis=0)
x_max = np.max(X, axis=0)
x_min = np.min(X, axis=0)

X_scaled = (X - mu) / (x_max - x_min)

In [31]:
num_centers = 15 

indices = np.random.choice(len(X_scaled), num_centers, replace=False)
centers = X_scaled[indices]

In [38]:
# 1. Build the Hidden Layer Matrix H (Samples x Centers)
sig = 1.0
H_gauss = np.zeros((len(X_scaled), num_centers))
H_multi = np.zeros((len(X_scaled), num_centers))
H_inv = np.zeros((len(X_scaled), num_centers))

for i in range(len(X_scaled)):
    for j in range(num_centers):
        H_gauss[i, j] = gauss(X_scaled[i], centers[j], sig)
        H_multi[i, j] = multi_quad(X_scaled[i], centers[j], sig)
        H_inv[i, j] = inv_multi_quad(X_scaled[i], centers[j], sig)


In [39]:
H_plus_bias_gauss = np.column_stack([np.ones(len(H_gauss)), H_gauss])
H_plus_bias_mul = np.column_stack([np.ones(len(H_multi)), H_multi])
H_plus_bias_inv = np.column_stack([np.ones(len(H_inv)), H_inv])

# 3. Solve for weights using the Pseudo-Inverse (Manual Training)
# W = (H^T * H)^-1 * H^T * y
weights_guass = np.linalg.pinv(H_plus_bias_gauss) @ Y
weights_mul = np.linalg.pinv(H_plus_bias_mul) @ Y
weights_inv = np.linalg.pinv(H_plus_bias_inv) @ Y

In [41]:
# Calculate final scores
scores_gauss = H_plus_bias_gauss @ weights_guass
scores_mult = H_plus_bias_mul @ weights_mul
scores_inv = H_plus_bias_inv @ weights_inv

# Convert scores to binary classes
predictions_gauss = (scores_gauss > 0.5).astype(int)
predictions_mul = (scores_mult > 0.5).astype(int)
predictions_inv = (scores_inv > 0.5).astype(int)

# Check Accuracy
accuracy_gauss = np.mean(predictions_gauss == Y)
accuracy_mul = np.mean(predictions_mul == Y)
accuracy_inv = np.mean(predictions_inv == Y)
print(f"Manual RBF Accuracy Gauss: {accuracy_gauss * 100:.2f}%")
print(f"Manual RBF Accuracy Multi: {accuracy_mul * 100:.2f}%")
print(f"Manual RBF Accuracy Inv: {accuracy_inv * 100:.2f}%")


Manual RBF Accuracy Gauss: 81.17%
Manual RBF Accuracy Multi: 83.51%
Manual RBF Accuracy Inv: 81.17%
